## Before you run this

**Training writes to `runs/`, never to `dataset/checkpoints/`.** Every config now sets
`overwrite: false`, so if the target directory already exists the run aborts with
`FileExistsError` instead of silently deleting a published checkpoint. Rename or remove
the directory under `runs/` when you want to re-run.

**`Run All` trains from scratch.** The teacher takes 75 epochs, the student up to 1000
(with early stopping). To only check that the pipeline is wired up, run the cells up to
the first validation and interrupt, or lower `epochs` in the config.

**Hyperparameter search is off** (`USE_OPTUNA = False`). Turning it on re-runs the
Optuna study, which took hours per notebook.


In [ ]:
# ── Where this run writes ────────────────────────────────────────────────────
# Output goes to runs/<RUN_NAME>/. dataset/checkpoints/ is never touched.
#
# OVERWRITE = False  -> if runs/<RUN_NAME> already exists, a new versioned
#                       directory is used (runs/<RUN_NAME>_002, _003, ...).
#                       Nothing is ever deleted.
# OVERWRITE = True   -> reuse runs/<RUN_NAME> and wipe it first.
import os

RUN_NAME  = "run"
OVERWRITE = False

RUN_DIR = os.path.join("../../runs", RUN_NAME)
if os.path.exists(RUN_DIR) and not OVERWRITE:
    _n = 2
    while os.path.exists(f"{RUN_DIR}_{_n:03d}"):
        _n += 1
    RUN_DIR = f"{RUN_DIR}_{_n:03d}"
print("this run writes to:", os.path.normpath(os.path.abspath(RUN_DIR)))

# Which weights the evaluation cells load:
#   "published" -> dataset/checkpoints/...  (reproduce the reported numbers)
#   "mine"      -> RUN_DIR                  (evaluate what you just trained)
EVAL_CKPT = "published"

TRAIN_CKPT = os.path.join(RUN_DIR, "tssi75_cslr_best.pt")


# CSRL — 75-keypoint Skeleton CSLR Runner

Modular runner notebook. All logic lives in `.py` modules:
- `config.py` — device, paths, CFG
- `skeleton.py` — DFS graph, TSSI generation
- `vocab.py` — vocabulary with gloss merge
- `dataset.py` — TSSIDataset, collate, dataloaders
- `model.py` — PoseNetworkCTC
- `losses.py` — CTC loss, decoding, WER
- `training.py` — two-phase training with resume checkpoints
- `optuna_search.py` — hyperparameter search

## 0 — Imports and setup

In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np

from config import DEVICE, AMP, DATA_DIR, CKPT_PATH, CFG
from vocab import build_vocab_from_raw
from dataset import load_pkl, make_dataloaders
from model import PoseNetworkCTC
from training import run_training, evaluate

print(f"device: {DEVICE} | amp: {AMP}")
print(f"CFG: {CFG}")

device: cuda | amp: True
CFG: {'num_joints': 75, 'hidden_dim': 256, 'tcn_blocks': 3, 'num_layers': 3, 'attn_heads': 4, 'dropout': 0.317638942624273, 'drop_path_rate': 0.1, 'batch_size': 16, 'grad_accum': 1, 'num_epochs': 75, 'early_stopping_patience': 20, 'phase1_epochs': 15, 'phase1_lr': 0.0001619145621568752, 'phase2_lr_backbone': 0.00021884694140182422, 'phase2_lr_head': 0.0005038675815696706, 'weight_decay': 0.0002465667957406484, 'grad_clip': 3.3943208852931304, 'ctc_smoothing': 0.10231111042825411, 'prior_beta': 0.32404536014294083, 'beam_width': 25, 'max_frames': 400, 'augment': True}


## 1 — Load data and build vocabulary

In [2]:
import os

train_raw = load_pkl(os.path.join(DATA_DIR, "Phoenix-2014T.train"))
dev_raw   = load_pkl(os.path.join(DATA_DIR, "Phoenix-2014T.dev"))
test_raw  = load_pkl(os.path.join(DATA_DIR, "Phoenix-2014T.test"))

print("Building vocabulary with gloss merge...")
V = build_vocab_from_raw(train_raw, dev_raw, test_raw)
NUM_CLASSES = V["num_classes"]
LOG_PRIOR = V["log_prior"].to(DEVICE)
i2g = V["i2g"]

train_loader, dev_loader, test_loader, _, _, _ = make_dataloaders(
    DATA_DIR, V["gloss_to_ids"], CFG
)
print(f"vocab: {NUM_CLASSES} classes")

Building vocabulary with gloss merge...
  merge map: 68 entries (0 hyphen-safe) | vocab: 1022 classes (<blank>=0, <unk>=1, gloss_train=1020)
  7096 samples | augment=True
  519 samples | augment=False
  642 samples | augment=False
vocab: 1022 classes


## 2 — Instantiate model

In [3]:
model = PoseNetworkCTC(
    num_classes=NUM_CLASSES,
    in_channels=CFG["num_joints"] * 3,
    hidden_dim=CFG["hidden_dim"],
    tcn_blocks=CFG["tcn_blocks"],
    lstm_layers=CFG["num_layers"],
    dropout=CFG["dropout"],
    drop_path_rate=CFG["drop_path_rate"],
    attn_heads=CFG["attn_heads"],
).to(DEVICE)

print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")

parameters: 6.59 M


## 3 — (Optional) Optuna hyperparameter search

Set `USE_OPTUNA = True` to run. Results are saved to `optuna_csrl_progress.json` and auto-resumed.

In [4]:
USE_OPTUNA = False   # set True to re-run the hyperparameter search (hours)

if USE_OPTUNA:
    from optuna_search import run_optuna_search

    # Screens configs on the FIRST 28 epochs of the full CFG["num_epochs"] run:
    # the cosine LR schedule is built on the full horizon, so early-epoch LR
    # matches the real run. Delete optuna_csrl_progress.json for a fresh search.
    best_params = run_optuna_search(
        cfg=CFG,
        train_loader=train_loader,
        dev_loader=dev_loader,
        num_classes=NUM_CLASSES,
        log_prior=LOG_PRIOR,
        device=DEVICE,
        amp=AMP,
        n_trials=4,
        trial_train_epochs=28,
    )
    for k, v in best_params.items():
        CFG[k] = v
    CFG["phase2_lr_backbone"] = CFG["phase2_lr_head"] * 0.5
    print("CFG updated with Optuna best.")
else:
    print("Optuna search disabled — using current CFG values.")

# always rebuild model with fresh weights (Optuna may have left dirty weights in memory)
model = PoseNetworkCTC(
    num_classes=NUM_CLASSES,
    in_channels=CFG["num_joints"] * 3,
    hidden_dim=CFG["hidden_dim"],
    tcn_blocks=CFG["tcn_blocks"],
    lstm_layers=CFG["num_layers"],
    dropout=CFG["dropout"],
    drop_path_rate=CFG["drop_path_rate"],
    attn_heads=CFG["attn_heads"],
).to(DEVICE)
print(f"fresh model: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M params")

[Optuna] RESUMED — 4/4 trials loaded from optuna_csrl_progress.json
  trial 1 | WER 67.72% | p1_lr=0.000118 p2_lr_h=0.000722 drop=0.3830 wd=0.000158
  trial 2 | WER 83.16% | p1_lr=0.000072 p2_lr_h=0.000138 drop=0.2145 wd=0.000540
  trial 3 | WER 67.72% | p1_lr=0.000118 p2_lr_h=0.000722 drop=0.3830 wd=0.000158
  trial 4 | WER 83.16% | p1_lr=0.000072 p2_lr_h=0.000138 drop=0.2145 wd=0.000540
  trials       : 4 (0 remaining)
  horizon      : 75 epochs (phase1 15 + phase2 60) — LR schedule built on this
  train/trial  : 28 epochs (phase1 15 + phase2 13) — early screening
  searching    : phase1_lr, phase2_lr_head, dropout, weight_decay
  progress     : optuna_csrl_progress.json
All trials already completed.

[Optuna] SEARCH COMPLETE — summary:
  #    WER%      p1_lr    p2_lr_h    drop         wd   time
----------------------------------------------------------------------
  1   67.72   0.000118   0.000722  0.3830   0.000158  5354s
  3   67.72   0.000118   0.000722  0.3830   0.000158  5174s


/home/ebufi/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 4 — Train (two-phase with resume checkpoints)

In [ ]:
BEST_FALLBACK = ""  # set path to an existing .pt to use as fallback, or "" to ignore

os.makedirs(RUN_DIR, exist_ok=True)
hist, best_wer = run_training(
    model=model,
    train_loader=train_loader,
    dev_loader=dev_loader,
    cfg=CFG,
    device=DEVICE,
    amp=AMP,
    log_prior=LOG_PRIOR,
    ckpt_path=TRAIN_CKPT,
    best_fallback_path=BEST_FALLBACK,
    fresh_start=True,  # set False to resume from checkpoint
)

   fresh_start=True — ignoring all checkpoints, training from epoch 0
PHASE 1: 15 epochs, FROZEN BACKBONE
  backbone frozen — trainable: 4,864,510


/home/ebufi/venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


[F1] Ep   1 | loss 20.057 | val 5.476 | dev WER 100.00% (S0 D3748 I0) | 191s
   -> best 100.00%
[F1] Ep   2 | loss 5.425 | val 5.288 | dev WER 100.00% (S0 D3748 I0) | 189s
[F1] Ep   3 | loss 5.301 | val 5.122 | dev WER 100.00% (S0 D3748 I0) | 189s
[F1] Ep   4 | loss 5.173 | val 5.049 | dev WER 99.81% (S0 D3741 I0) | 190s
   -> best 99.81%


/home/ebufi/venv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


[F1] Ep   5 | loss 5.058 | val 4.858 | dev WER 99.31% (S6 D3716 I0) | 189s
   -> best 99.31%
   [resume ckpt] saved @ phase=1 gep=5
[F1] Ep   6 | loss 4.973 | val 4.887 | dev WER 99.39% (S3 D3722 I0) | 189s
[F1] Ep   7 | loss 4.902 | val 4.716 | dev WER 99.12% (S12 D3703 I0) | 188s
   -> best 99.12%
[F1] Ep   8 | loss 4.837 | val 4.687 | dev WER 98.83% (S11 D3693 I0) | 187s
   -> best 98.83%
[F1] Ep   9 | loss 4.776 | val 4.682 | dev WER 98.75% (S30 D3671 I0) | 188s
   -> best 98.75%
[F1] Ep  10 | loss 4.729 | val 4.627 | dev WER 98.37% (S43 D3644 I0) | 188s
   -> best 98.37%
   [resume ckpt] saved @ phase=1 gep=10
[F1] Ep  11 | loss 4.692 | val 4.618 | dev WER 98.16% (S68 D3611 I0) | 187s
   -> best 98.16%
[F1] Ep  12 | loss 4.661 | val 4.598 | dev WER 97.89% (S58 D3611 I0) | 186s
   -> best 97.89%
[F1] Ep  13 | loss 4.638 | val 4.579 | dev WER 97.81% (S67 D3599 I0) | 188s
   -> best 97.81%
[F1] Ep  14 | loss 4.624 | val 4.562 | dev WER 97.76% (S76 D3588 I0) | 186s
   -> best 97.76%
[

## 5 — Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(hist["ep"], hist["loss"], marker="o", ms=3, label="train (CTC + SR-CTC)")
if hist.get("val_loss") and any(v == v for v in hist["val_loss"]):  # any non-NaN
    ax1.plot(hist["ep"], hist["val_loss"], marker="o", ms=3,
             color="tab:orange", label="dev (main CTC)")
ax1.set_title("Loss")
ax1.set_xlabel("epoch")
ax1.set_ylabel("loss")
ax1.grid(alpha=0.3)
ax1.legend()

if hist["wer"]:
    bi = int(np.argmin(hist["wer"]))
    ax2.plot(hist["ep"], hist["wer"], marker="o", ms=3, color="crimson")
    ax2.scatter([hist["ep"][bi]], [hist["wer"][bi]], color="k", zorder=5,
               label=f"best {hist['wer'][bi]:.2f}%")
    if 2 in hist["phase"]:
        f2 = hist["ep"][hist["phase"].index(2)]
        ax1.axvline(f2, ls="--", c="gray", alpha=0.6)
        ax2.axvline(f2, ls="--", c="gray", alpha=0.6, label="Phase 2 start")
    ax2.legend()

ax2.set_title("Dev WER (corpus-level)")
ax2.set_xlabel("epoch")
ax2.set_ylabel("WER %")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("tssi75_curves.png", dpi=110)
plt.show()

## 6 — Test set evaluation (greedy vs beam)

In [ ]:
_eval_path = CKPT_PATH if EVAL_CKPT == "published" else TRAIN_CKPT
print("evaluating:", _eval_path)
ck = torch.load(_eval_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ck["model"])
model.to(DEVICE).eval()
print(f"checkpoint epoch {ck['epoch']} | dev WER {ck['wer']*100:.2f}%\n")

wer_g, dg, _ = evaluate(model, test_loader, CFG, DEVICE, LOG_PRIOR, use_beam=False)
print(f"TEST greedy      WER {wer_g*100:.2f}%  (S{dg['S']} D{dg['D']} I{dg['I']} N{dg['N']})")

wer_b, db, _ = evaluate(model, test_loader, CFG, DEVICE, LOG_PRIOR, use_beam=True)
print(f"TEST beam({CFG['beam_width']})    WER {wer_b*100:.2f}%  (S{db['S']} D{db['D']} I{db['I']} N{db['N']})")

print(f"\n>>> WER to report = {min(wer_g, wer_b)*100:.2f}%")